# Mistério em João Pessoa

Adaptação do lendário [SQL Murder Mystery](https://github.com/NUKnightLab/sql-mysteries) (Knight Lab / Northwestern University, conteúdo original sob licença CC BY-SA 4.0). Nesta adaptação você resolve tudo com pandas + MinIO. A cidade e alguns nomes/ruas viraram locais de João Pessoa.

Um assassinato foi registrado em **João Pessoa** em **15/01/2018**. A polícia recolheu 6 arquivos crus (`dados/*.csv`) e é com eles que você vai trabalhar.

## O que você recebeu

| Arquivo | Colunas | O que é |
|---|---|---|
| `ocorrencia.csv` | `data, tipo, descricao, cidade` | Boletins de Ocorrência |
| `pessoa.csv` | `id, nome, detran_id, numero_endereco, rua, cpf` | Cadastro de Pessoas |
| `detran.csv` | `id, idade, altura, cor_olhos, cor_cabelo, genero, placa, marca_veiculo, modelo_veiculo` | Cadastro do DETRAN |
| `depoimento.csv` | `pessoa_id, relato` | Depoimentos |
| `membro_academia.csv` | `id, pessoa_id, nome, data_matricula, plano` | Matrículas da Academia |
| `checkin_academia.csv` | `matricula_id, data_checkin, hora_entrada, hora_saida` | Check-ins da Academia |

## O que você entrega

1. **Fase 1 — Bronze**: os 6 CSVs publicados como tabelas bronze (`df.to_parquet(f"s3://{BUCKET}/bronze/<nome>.parquet", storage_options=STORAGE_OPTIONS)`).
2. **Fase 2 — Investigação**: livre — use pandas (`pd.read_parquet` + `merge`/filtros) para seguir as pistas até chegar a **1** suspeito.
3. **Fase 3 — Resposta final na Silver**: uma tabela `silver.resposta_caso` com sua conclusão (ver Fase 3 no final deste notebook pro formato esperado).

In [ ]:
import os

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Conexão com o MinIO (S3-compatível) — pandas usa isso direto via s3fs
BUCKET = os.environ.get("LAKEHOUSE_S3_BUCKET", "lakehouse")
STORAGE_OPTIONS = {
    "key": os.environ.get("LAKEHOUSE_S3_ACCESS_KEY", "trilha"),
    "secret": os.environ.get("LAKEHOUSE_S3_SECRET_KEY", "trilha123"),
    "client_kwargs": {"endpoint_url": os.environ.get("LAKEHOUSE_S3_ENDPOINT", "http://minio:9000")},
}

## Fase 1 — Bronze

O padrão pra publicar qualquer CSV cru como tabela bronze é sempre o mesmo — 2 passos, só pandas:

```python
df = pd.read_csv("dados/<arquivo>.csv")                                                                     # lê o CSV cru direto do disco
df.to_parquet(f"s3://{BUCKET}/bronze/<nome_tabela>.parquet", storage_options=STORAGE_OPTIONS, index=False)  # publica: grava Parquet
```

Cada tabela vira **1 arquivo Parquet** na camada — `bronze/<nome_tabela>.parquet`, dá pra conferir pelo MinIO Console (http://localhost:9001) — os arquivos vão aparecendo em `lakehouse/bronze/`.

Um exemplo pronto (`ocorrencia`), com uma ilustração rápida de como ler de volta logo depois — daí é sua vez de fazer o mesmo padrão para as outras 5 tabelas.

In [ ]:
# Exemplo pronto: ocorrencia
df_ocorrencia = pd.read_csv("dados/ocorrencia.csv")
df_ocorrencia.to_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_ocorrencia

### Lendo de volta (exemplo rápido)

Pra ler uma tabela publicada é só apontar o `pd.read_parquet` pro mesmo caminho, ilustrado com a tabela que acabamos de publicar:

- `pd.read_parquet(f"s3://{BUCKET}/bronze/<tabela>.parquet", storage_options=STORAGE_OPTIONS)` lê o Parquet inteiro direto do MinIO com pandas — é o que você vai usar na Fase 2 pra ler cada uma das 6 tabelas bronze.

Pra ver o que já existe fisicamente numa camada, sem precisar ler o conteúdo nem escrever código nenhum: abra o **MinIO Console** (http://localhost:9001) e olhe os arquivos em `lakehouse/bronze/`.

In [ ]:
# Lendo a tabela de volta direto do MinIO com pandas — deve ser idêntica a df_ocorrencia
pd.read_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS)

In [ ]:
df_pessoa = pd.read_csv("dados/pessoa.csv")
df_pessoa.to_parquet(f"s3://{BUCKET}/bronze/pessoa.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_pessoa

In [ ]:
df_detran = pd.read_csv("dados/detran.csv")
df_detran.to_parquet(f"s3://{BUCKET}/bronze/detran.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_detran

In [ ]:
df_depoimento = pd.read_csv("dados/depoimento.csv")
df_depoimento.to_parquet(f"s3://{BUCKET}/bronze/depoimento.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_depoimento

In [ ]:
df_membro_academia = pd.read_csv("dados/membro_academia.csv")
df_membro_academia.to_parquet(f"s3://{BUCKET}/bronze/membro_academia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_membro_academia

In [ ]:
df_checkin_academia = pd.read_csv("dados/checkin_academia.csv")
df_checkin_academia.to_parquet(f"s3://{BUCKET}/bronze/checkin_academia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_checkin_academia

Checagem: as 6 tabelas devem aparecer em `lakehouse/bronze/` no **MinIO Console** (http://localhost:9001).

## Fase 2 — Investigação (pandas)

A partir de agora é livre: leia as tabelas bronze direto do MinIO com `pd.read_parquet(f"s3://{BUCKET}/bronze/<tabela>.parquet", storage_options=STORAGE_OPTIONS)`, e siga as pistas com `merge`/filtros de DataFrame, do jeito que preferir.

Um roteiro sugerido (não obrigatório seguir exatamente esta ordem, mas ajuda a não se perder):

1. Pesquise a ocorrência e bus as testemunhas em `pessoa`.
2. Pesquise o `depoimento` das respectivas testemunhas.
3. Cada depoimento traz uma pista diferente — uma aponta para `membro_academia` a outra para `detran`.
4. Pesquise pela pistas, mencionadas pelas testemunhas, nas tabelas correspondentes.
5. Confirme em `checkin_academia` que o suspeito tem um check-in na academia na data que a segunda testemunha mencionou.

In [ ]:
# Lendo todas as tabelas bronze direto do MinIO via pandas
ocorrencia = pd.read_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS)

### Passo 1 — a ocorrência

Um assassinato foi registrado em **João Pessoa** em **15/01/2018**.

In [ ]:
# TODO:

### Passo 2 — as testemunhas

In [ ]:
# TODO: 

### Passo 3 — duas pistas, duas tabelas


In [ ]:
# TODO:

In [ ]:
# TODO:

### Passo 4 — cruzando as pistas

In [ ]:
# TODO

### Passo 5 — confirmar com o check-in

In [ ]:
# TODO

## Fase 3 — Resposta final na Silver

Chegou a hora de publicar sua conclusão como uma tabela — o entregável desta tarefa. Monte um DataFrame de **1 linha** com estas colunas:

| coluna | conteúdo |
|---|---|
| `nome_suspeito` | o nome completo da pessoa em `pessoa` |
| `placa_veiculo` | a placa (de `detran`) que fechou o caso |
| `pista_academia` | qual detalhe da matrícula (início do `id` + status do plano) bateu com o depoimento |
| `pista_veiculo` | qual trecho da placa bateu com o depoimento |
| `justificativa` | 1-2 frases explicando o raciocínio (pode ser texto livre) |

E publique com `df_resposta.to_parquet(f"s3://{BUCKET}/silver/resposta_caso.parquet", storage_options=STORAGE_OPTIONS, index=False)` — depois disso, o **MinIO Console** (http://localhost:9001), em `lakehouse/silver/resposta_caso.parquet`, já mostra o resultado.

In [ ]:
# TODO: monte df_resposta (1 linha, colunas da tabela acima) e publique na silver

---

Terminou? `lakehouse/bronze/` deve ter as 6 tabelas desta tarefa, e `lakehouse/silver/resposta_caso.parquet` deve ter sua conclusão — dá pra confirmar tudo pelo MinIO Console (http://localhost:9001) sem precisar de mais nada.